# Pedal Poll analysis

Processing steps before analysis
- [x] group sessions during the same count window 
    - check for same location and count date, then check for session start time within the same pedal poll counting block (i.e. for 7-9 count, start times at hour 6, 7, or 8)
    - sum counts assumed to be in the same session as well as the elapsed time for each count
- [x] Check how many sites had overlapping counts for the same count window
- [x] group counts where multiple people counted in opposite directions
    - this should be captured in the above steps, but what about summing the time?
- [x] Add a field for weekday vs weekend and to to track count session: "At each official count location, the goal was for 120-minute long counts during four weekday periods (07–09:00, 11–13:00, 14–16:00 and 16–18:00) and one weekend period (12–14:00)."

Tests:
- [x] same number of total counts after grouping - none of these steps should remove counts
- [x] same number of counterpoints
- [ ] possibly check short counts - do they belong in any included sessions? 

In [1]:
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm

In [2]:
pd.options.mode.copy_on_write = True

## Load cleaned data

In [3]:
data_2021_url = "https://velocanadabikes.org/wp-content/uploads/2022/Pedal%20Poll%202021%20Data%20UPLOAD.csv"
data_2021 = pd.read_csv(data_2021_url, index_col = 0)
data_2022 = pd.read_csv("data/2022_cleaned_data.csv", low_memory=False) # to avoid dtype warning
data_2023 = pd.read_csv("data/2023_cleaned_data.csv")
data_2024 = pd.read_csv("data/2024_cleaned_data.csv", low_memory=False) # to avoid dtype warning

counterpoint_id_groups = pd.read_csv("data/counterpoint_id_groupings.csv")

In [4]:
data_2024.columns

Index(['counterpoint_id', 'counterpoint_name', 'lat', 'lng', 'coordinates',
       'geocoded_city', 'geocoded_province', 'survey_session_id',
       'start_datetime', 'end_datetime', 'start_datetime_year',
       'start_datetime_month', 'start_datetime_month_number',
       'start_datetime_dayofmonth', 'start_datetime_dayofweek',
       'start_datetime_dayofweek_number', 'start_datetime_hourofday',
       'elapsed_time_seconds', 'elapsed_time_minutes', 'team_id', 'team_name',
       'count_time', 'easy_bike_total', 'easy_pedestrian_total',
       'easy_personal_vehicle_total', 'easy_oversized_total', 'date',
       'location_notes', 'notes'],
      dtype='object')

In [5]:
group_columns = ['year', 'grouped_counterpoint_id', 'geocoded_city', 'geocoded_province', 'start_datetime_dayofmonth', 
                                             'count_session_label']

In [6]:
data_2023 = data_2023[~(data_2023['end_datetime'] == 'Not listed ')] # remove count session without end time

In [7]:
data_2021['easy_bike_total'] = 1 # every row is a cyclist counted
# check number of paper counts
data_2021[data_2021['start_datetime_dayofmonth'].isna()].groupby('geocoded_city')['easy_bike_total'].sum().reset_index()

,geocoded_city,easy_bike_total
0,Burnaby,60
1,Charlottetown,13
2,Dartmouth,7
3,Dieppe,196
4,Edmonton,446
5,Fredericton,12
6,Guelph,66
7,Halifax,97
8,Kelowna,465
9,Kingston,56


## Make fields to identify weekday vs weekend and time of day of count

In [8]:
from functions import identify_count_session

In [9]:
# check for weekend
# redo the day of week operation to include the hand count data
for df in [data_2021, data_2022, data_2023, data_2024]:
    df['is_weekend'] = pd.to_datetime(df['start_datetime'], format='mixed').dt.day_name().isin(['Saturday', 'Sunday'])
    df = identify_count_session(df)

Names for the count sessions:
'weekday_morning': 7-9 am
'weekday_lunch': 11 am - 1 pm
'weekday_afternoon': 2-4 pm
'weekday_late_afternoon': 4-6 pm
'weekend_afternoon': 12-2 pm


Check 'not_assigned' sessions and see if we need to change anything

In [10]:
# the first one might be fine, it's ~11-1 instead of 12-2. For the simplicitly of analysis, we will exclude it.
# the last one is at the right time but not at a valid counterpoint.
data_2024[data_2024['count_session_label'] == 'not_assigned'][['counterpoint_id', 'counterpoint_name', 'geocoded_city', 'geocoded_province', 
                                                               'survey_session_id', 'start_datetime', 'end_datetime','start_datetime_dayofweek',
                                                               'elapsed_time_minutes', 'is_weekend', 'count_session_label']].drop_duplicates()

,counterpoint_id,counterpoint_name,geocoded_city,geocoded_province,survey_session_id,start_datetime,end_datetime,start_datetime_dayofweek,elapsed_time_minutes,is_weekend,count_session_label
7264,5381.0,Stittsville Trans Canada Trail (Eastbound coun...,Goulbourn,Ontario,34985,2024-06-08 10:55:47,2024-06-08 12:57:01,Saturday,121.0,True,not_assigned
14538,8222.0,Bay Facing East 4-6,Hamilton,Ontario,34798,2024-06-04 18:15:27,2024-06-04 18:22:01,Tuesday,6.0,False,not_assigned
29283,5311.0,Erin Mills (Burnhamthorpe Trail),Mississauga,Ontario,34569,2024-06-04 09:42:47,2024-06-04 09:55:31,Tuesday,12.0,False,not_assigned
34068,NaN,14th Avenue & Pasqua Street in Regina.,Regina,Saskatchewan,14,2024-06-08 16:00:00,2024-06-08 18:00:00,NaN,NaN,True,not_assigned


In [11]:
# these all seem to not match an official count time; some might have the paper count time entered incorrectly
data_2023[data_2023['count_session_label'] == 'not_assigned'][['counterpoint_id', 'counterpoint_name', 'geocoded_city', 'geocoded_province', 
                                                               'survey_session_id', 'start_datetime', 'end_datetime','start_datetime_dayofweek',
                                                               'elapsed_time_minutes', 'is_weekend', 'count_session_label']].drop_duplicates()

,counterpoint_id,counterpoint_name,geocoded_city,geocoded_province,survey_session_id,start_datetime,end_datetime,start_datetime_dayofweek,elapsed_time_minutes,is_weekend,count_session_label
1934,5286.0,Cavan Larose,Ottawa,Ontario,32123,2023-06-10 19:51:19,2023-06-10 20:04:39,Saturday,13.0,True,not_assigned
6619,7339.0,Covered bridge,Guelph,Ontario,32061,2023-06-10 10:44:55,2023-06-10 12:45:13,Saturday,120.0,True,not_assigned
6726,5556.0,McRae Street (Gow's Bridge),Guelph,Ontario,32060,2023-06-10 10:34:00,2023-06-10 12:34:11,Saturday,120.0,True,not_assigned
6834,7338.0,Hillside Dr. N.,Elliot Lake,Ontario,32058,2023-06-10 09:24:30,2023-06-10 11:24:20,Saturday,119.0,True,not_assigned
6954,6606.0,Boat House,Guelph,Ontario,32052,2023-06-10 07:53:02,2023-06-10 09:57:47,Saturday,124.0,True,not_assigned
11299,7285.0,Waterloo,Guelph,Ontario,31916,2023-06-07 09:11:45,2023-06-07 09:42:05,Wednesday,30.0,False,not_assigned
15513,6499.0,Hillside and Spine,Elliot Lake,Ontario,31691,2023-06-06 09:32:59,2023-06-06 11:04:03,Tuesday,91.0,False,not_assigned
19389,NaN,82 st - 118 ave,Edmonton,Alberta,36,2023-06-10 09:20:00,2023-06-10 11:20:00,NaN,NaN,True,not_assigned
19401,5391.0,Riverfront Park,Moncton,New Brunswick,48,2023-06-07 02:07:00,2023-06-07 04:07:00,NaN,NaN,False,not_assigned
19421,NaN,Not written but came from same email as above,Fredericton,New Brunswick,69,2023-06-12 02:00:00,2023-06-12 16:00:00,NaN,NaN,False,not_assigned


In [12]:
# these also seem like genuine mismatches
data_2022[data_2022['count_session_label'] == 'not_assigned'][['counterpoint_id', 'geocoded_city', 'geocoded_province', 
                                                               'survey_session_id', 'start_datetime', 'end_datetime','start_datetime_dayofweek',
                                                               'elapsed_time_seconds', 'is_weekend', 'count_session_label']].drop_duplicates()

,counterpoint_id,geocoded_city,geocoded_province,survey_session_id,start_datetime,end_datetime,start_datetime_dayofweek,elapsed_time_seconds,is_weekend,count_session_label
37,6542.0,Montréal,Québec,4177.0,2022-06-14 18:01,2022-06-14 18:31,Tuesday,1800,False,not_assigned
1763,5676.0,Regina,Saskatchewan,4162.0,2022-06-11 17:22,2022-06-11 17:58,Saturday,2160,True,not_assigned
1783,6542.0,Montréal,Québec,4161.0,2022-06-11 17:18,2022-06-11 18:23,Saturday,3900,True,not_assigned
1926,6542.0,Montréal,Québec,4157.0,2022-06-11 17:11,2022-06-11 17:16,Saturday,300,True,not_assigned
1949,5270.0,Calgary,Alberta,4156.0,2022-06-11 14:40,2022-06-11 15:11,Saturday,1860,True,not_assigned
3272,5749.0,Winnipeg,Manitoba,4136.0,2022-06-11 14:31,2022-06-11 15:29,Saturday,3480,True,not_assigned
5299,5749.0,Winnipeg,Manitoba,4104.0,2022-06-11 14:06,2022-06-11 14:21,Saturday,900,True,not_assigned
7345,6391.0,Brampton,Ontario,4038.0,2022-06-11 14:00,2022-06-11 14:11,Saturday,660,True,not_assigned
8785,5195.0,Halifax,Nova Scotia,3919.0,2022-06-11 14:37,2022-06-11 14:59,Saturday,1320,True,not_assigned
12137,6542.0,Montréal,Québec,3850.0,2022-06-11 10:37,2022-06-11 11:37,Saturday,3600,True,not_assigned


In [13]:
# some of these look like continuations of valid counts, but because they are so short they are marked as not starting at the right time
# eg counterpoint ID 5585 which has a bunch of starts and stops
# in that example it is probably fine to discard the out-of-window data since it starts before the scheduled start time anyway
data_2021[data_2021['count_session_label'] == 'not_assigned'][['counterpoint_id', 'geocoded_city', 'geocoded_province', 
                                                               'survey_session_id', 'start_datetime', 'end_datetime','start_datetime_dayofweek',
                                                               'elapsed_time_seconds', 'is_weekend', 'count_session_label']].drop_duplicates()

,counterpoint_id,geocoded_city,geocoded_province,survey_session_id,start_datetime,end_datetime,start_datetime_dayofweek,elapsed_time_seconds,is_weekend,count_session_label
1543,5655,Champlain,Ontario,2466,2021-06-02 18:10,2021-06-02 18:18,Wednesday,480,False,not_assigned
1645,5616,Saskatoon,Saskatchewan,2454,2021-06-05 16:06,2021-06-05 17:06,Saturday,3600,True,not_assigned
1768,5666,Oakville,Ontario,2453,2021-06-05 17:27,2021-06-05 18:58,Saturday,5460,True,not_assigned
1778,5752,Edmonton,Alberta,2452,2021-06-05 14:56,2021-06-05 15:40,Saturday,2640,True,not_assigned
5570,5541,Toronto,Ontario,2397,2021-06-05 14:43,2021-06-05 15:59,Saturday,4560,True,not_assigned
7108,5657,St. Catharines,Ontario,2351,2021-06-05 14:08,2021-06-05 14:51,Saturday,2580,True,not_assigned
7189,5541,Toronto,Ontario,2348,2021-06-05 14:00,2021-06-05 14:40,Saturday,2400,True,not_assigned
9851,5542,Toronto,Ontario,2316,2021-06-05 14:00,2021-06-05 14:07,Saturday,420,True,not_assigned
10593,5745,Capilano No. 5,British Columbia,2220,2021-06-05 10:54,2021-06-05 10:59,Saturday,300,True,not_assigned
13904,5731,Windsor,Ontario,2165,2021-06-05 10:39,2021-06-05 12:46,Saturday,7620,True,not_assigned


In [14]:
# add grouped counterpoint id and create combined dataset
data_2021['easy_bike_total'] = 1 # every row is a cyclist counted
data_2022['easy_bike_total'] = 1
data_2021['year'] = 2021
data_2022['year'] = 2022
data_2023['year'] = 2023
data_2024['year'] = 2024


In [15]:
# fix Nan in paper count data
data_2021['start_datetime_dayofmonth'] = pd.to_datetime(data_2021['start_datetime']).dt.day

In [16]:
all_data = pd.concat([data_2021, data_2022, data_2023, data_2024]) # this fills missing columns with NA
original_shape = all_data.shape[0]
rows_removed = 0

# add grouped counterpoint_id
# use city identified by the api and saved in counterpoint_id_groups
rows_removed += all_data[all_data['counterpoint_id'].isna()].shape[0]
all_data = all_data[~all_data['counterpoint_id'].isna()] #remove NA counterpoint IDs
all_data = all_data.drop(columns = ['coordinates', 'geocoded_city']).merge(counterpoint_id_groups[['counterpoint_id',
                                                    'grouped_counterpoint_id', 'geocoded_city']].drop_duplicates(),
                                                    on = 'counterpoint_id', how = 'left')

### Clean city and province names

In [17]:
official_sites = pd.read_csv("data/counterpoint_official_sites_multiyear_counts.csv")
official_site_ids = official_sites[official_sites['official_2021_site'] == 'yes']['grouped_counterpoint_id'].unique()

In [18]:
# remove whitespace
all_data['geocoded_city'] = all_data['geocoded_city'].str.strip()

In [19]:
# remove the accent in quebec
all_data.loc[all_data['geocoded_province'] == 'Québec', 'geocoded_province'] = 'Quebec'

In [20]:
all_data['geocoded_city'].sort_values().unique()

array(['Area D (Roberts Creek)', 'Area E (Elphinstone)',
       'Area H (The Slocan Valley)', 'Banff', 'Brampton', 'Bridgewater',
       'Burlington', 'Burnaby', 'Calgary', 'Campbell River', 'Canmore',
       'Capilano No. 5', 'Champlain', 'Charlottetown', 'City of Leduc',
       'City of Saint John', 'Coldbrook', 'Coquitlam', 'Corner Brook',
       'Cumberland', 'Dartmouth', 'Dieppe', 'District of North Vancouver',
       'Edmonton', 'Elliot Lake', 'Fredericton', 'Gatineau', 'Glace Bay',
       'Goulbourn', 'Guelph', 'Halifax', 'Hamilton',
       'Head of Chezzetcook', 'Hebbville', 'Kamloops', 'Kelowna',
       'Kingston', 'Kitchener', 'London', 'Lunenburg', 'Maple Ridge',
       'Medicine Hat', 'Mississauga', 'Moncton', 'Montreal', 'Nelson',
       'New Glasgow', 'No city', 'North Dumfries', 'North Vancouver',
       'Oakville', 'Orillia', 'Ottawa', 'Paradise', 'Peterborough',
       'Port Coquitlam', 'Port Moody', 'Quebec City', 'Red Deer',
       'Regina', 'Richmond', 'Saanich', 'S

In [21]:
# group city names
all_data.loc[all_data['geocoded_city'] == 'City of Saint John', 'geocoded_city'] = 'Saint John'
all_data.loc[all_data['geocoded_city'] == 'Orl�ans', 'geocoded_city'] = 'Ottawa' # this example is in Ottawa
all_data.loc[all_data['geocoded_city'] == "St John's", 'geocoded_city'] = "St. John's"
all_data.loc[all_data['geocoded_city'] == 'District of North Vancouver', 'geocoded_city'] = 'North Vancouver'

In [22]:
len(all_data['geocoded_city'].unique())

79

In [23]:
len(all_data[all_data['year'] == 2024]['geocoded_city'].sort_values().unique())

35

In [24]:
all_data[all_data['year'] == 2024]['geocoded_city'].sort_values().unique()

array(['Calgary', 'Canmore', 'Charlottetown', 'Cumberland', 'Dartmouth',
       'Edmonton', 'Fredericton', 'Goulbourn', 'Guelph', 'Halifax',
       'Hamilton', 'Kelowna', 'Kitchener', 'London', 'Mississauga',
       'Moncton', 'Montreal', 'Nelson', 'No city', 'North Dumfries',
       'Ottawa', 'Peterborough', 'Red Deer', 'Regina', 'Richmond',
       'Saskatoon', "St. John's", 'Toronto', 'Vancouver', 'Waterloo',
       'Whitehorse', 'Windsor', 'Winnipeg', 'Wolfville', 'Yellowknife'],
      dtype=object)

In [25]:
focus_communities = ["Vancouver", "Toronto", "Ottawa", "Winnipeg", "London", "Halifax", "Kelowna", 
                     "Saskatoon", "Red Deer", "Whitehorse", "Mississauga", "Moncton",
                     "St. John's", "Charlottetown"]

In [26]:
# make sure at least the official sites have city names that make sense
all_data[all_data['grouped_counterpoint_id'].isin(official_site_ids)]['geocoded_city'].sort_values().unique()

array(['Charlottetown', 'Cumberland', 'Dartmouth', 'Dieppe', 'Goulbourn',
       'Halifax', 'Kelowna', 'London', 'Mississauga', 'Moncton',
       'No city', 'Ottawa', 'Red Deer', 'Saskatoon', "St. John's",
       'Toronto', 'Vancouver', 'Whitehorse', 'Winnipeg'], dtype=object)

In [27]:
all_data[(all_data['grouped_counterpoint_id'].isin(official_site_ids))
        & (all_data['geocoded_city'] == 'No city')][['grouped_counterpoint_id', 'counterpoint_name', 'geocoded_province']].drop_duplicates()

,grouped_counterpoint_id,counterpoint_name,geocoded_province
91081,5204.0,Pathway NE of Roundabout Adjacent to Longfield...,Ontario
104481,5401.0,Kanata - Watts Creek MUP,Ontario
115225,4448.0,Kildonan Drive,Manitoba
126997,5520.0,Des Meurons,Manitoba
148287,4448.0,Kildonan Drive between Greene Avenue and Leigh...,NaN


In [28]:
# group related cities that are focus communities
all_data.loc[all_data['geocoded_city'] == 'Kanata', 'geocoded_city'] = 'Ottawa'
all_data.loc[all_data['geocoded_city'] == 'Cumberland', 'geocoded_city'] = 'Ottawa' 
all_data.loc[all_data['geocoded_city'] == 'Goulbourn', 'geocoded_city'] = 'Ottawa' 
all_data.loc[all_data['geocoded_city'] == 'Dieppe', 'geocoded_city'] = 'Moncton'
all_data.loc[all_data['geocoded_city'] == 'Dartmouth', 'geocoded_city'] = 'Halifax'
all_data.loc[all_data['geocoded_city'] == 'Coquitlam', 'geocoded_city'] = 'Vancouver'
all_data.loc[all_data['geocoded_city'] == 'Capilano No. 5', 'geocoded_city'] = 'Vancouver'
all_data.loc[all_data['geocoded_city'] == 'Burnaby', 'geocoded_city'] = 'Vancouver'
all_data.loc[all_data['geocoded_city'] == 'Port Coquitlam', 'geocoded_city'] = 'Vancouver'
all_data.loc[all_data['geocoded_city'] == 'Port Moody', 'geocoded_city'] = 'Vancouver'
all_data.loc[all_data['geocoded_city'] == 'Saanich', 'geocoded_city'] = 'Victoria'
all_data.loc[all_data['geocoded_city'] == 'City of Leduc', 'geocoded_city'] = 'Edmonton'
all_data.loc[all_data['geocoded_city'] == 'Paradise', 'geocoded_city'] = "St. John's"
all_data.loc[all_data['geocoded_city'] == 'Hebbville', 'geocoded_city'] = 'Bridgewater'
all_data.loc[all_data['grouped_counterpoint_id'] == 5204, 'geocoded_city'] = 'Ottawa' # this one has multi-year counts
all_data.loc[all_data['grouped_counterpoint_id'] == 5401, 'geocoded_city'] = 'Ottawa' 
all_data.loc[all_data['grouped_counterpoint_id'] == 4448, 'geocoded_city'] = 'Winnipeg' 
all_data.loc[all_data['grouped_counterpoint_id'] == 5520, 'geocoded_city'] = 'Winnipeg' 

In [29]:
all_data['geocoded_city'].sort_values().unique()

array(['Area D (Roberts Creek)', 'Area E (Elphinstone)',
       'Area H (The Slocan Valley)', 'Banff', 'Brampton', 'Bridgewater',
       'Burlington', 'Calgary', 'Campbell River', 'Canmore', 'Champlain',
       'Charlottetown', 'Coldbrook', 'Corner Brook', 'Edmonton',
       'Elliot Lake', 'Fredericton', 'Gatineau', 'Glace Bay', 'Guelph',
       'Halifax', 'Hamilton', 'Head of Chezzetcook', 'Kamloops',
       'Kelowna', 'Kingston', 'Kitchener', 'London', 'Lunenburg',
       'Maple Ridge', 'Medicine Hat', 'Mississauga', 'Moncton',
       'Montreal', 'Nelson', 'New Glasgow', 'No city', 'North Dumfries',
       'North Vancouver', 'Oakville', 'Orillia', 'Ottawa', 'Peterborough',
       'Quebec City', 'Red Deer', 'Regina', 'Richmond', 'Saint John',
       'Sarnia', 'Saskatoon', 'Sechelt', 'St. Catharines', "St. John's",
       'Stouffville', 'Stratford', 'Toronto', 'Township of Langley',
       'Vancouver', 'Victoria', 'Waterloo', 'Whitehorse', 'Windsor',
       'Winnipeg', 'Wolfville', 'Ye

In [30]:
len(all_data['geocoded_city'].sort_values().unique())

66

In [31]:
assert all_data.shape[0] == data_2021.shape[0] + data_2022.shape[0] + data_2023.shape[0] + data_2024.shape[0] - rows_removed

In [32]:
all_data.to_csv("non_aggregated_cleaned_data.csv", index = None)

In [33]:
all_data.shape

(182386, 59)

## Data aggreggation

- [x] figure out how many count windows are present multi-year
- [x] aggregate count sessions by location, year, date, identified count session window (sum)
- [x] calculate count rate
- [x] include demographic data

In [34]:
from tqdm import tqdm

Find and diagnose errors in converting to numeric

In [35]:
# this count says their non-bike values are estimates. Should they be used, or should they be removed?
# I think used
index = all_data.loc[(pd.to_numeric(all_data['easy_personal_vehicle_total'],  errors = 'coerce').isna())
         & (all_data['year'] > 2022)].index

all_data.loc[(pd.to_numeric(all_data['easy_personal_vehicle_total'],  errors = 'coerce').isna())
         & (all_data['year'] > 2022)]

,counterpoint_id,counterpoint_name,lat,lng,geocoded_province,user_id,survey_session_id,elapsed_time_minutes,start_datetime,end_datetime,...,count_time,easy_pedestrian_total,easy_personal_vehicle_total,easy_oversized_total,location_notes,notes,counterpoint_start_datetime,time,grouped_counterpoint_id,geocoded_city
148282,6482.0,Morris st. at Barrington,NaN,NaN,Nova Scotia,NaN,61.0,NaN,2023-06-06 11:00:00,2023-06-06 13:00:00,...,NaN,~400,~1200,~36,NaN,these numbers (except for cyclists) are estima...,NaN,NaN,6482.0,Halifax


In [36]:
#use estimates as actual values
all_data.loc[index, 'easy_pedestrian_total'] = 400
all_data.loc[index, 'easy_personal_vehicle_total'] = 1200
all_data.loc[index, 'easy_oversized_total'] = 36

In [37]:
# confirm that all mode counts can be converted to numbers
assert all_data.loc[(pd.to_numeric(all_data['easy_oversized_total'],  errors = 'coerce').isna())
         & (all_data['year'] > 2022)].shape[0] == 0

assert all_data.loc[(pd.to_numeric(all_data['easy_pedestrian_total'],  errors = 'coerce').isna())
         & (all_data['year'] > 2022)].shape[0] == 0

assert all_data.loc[(pd.to_numeric(all_data['easy_personal_vehicle_total'],  errors = 'coerce').isna())
         & (all_data['year'] > 2022)].shape[0] == 0

In [38]:
# remove these - they are not in canada
all_data[all_data['grouped_counterpoint_id'].isna()][['year','counterpoint_id']].drop_duplicates()

,year,counterpoint_id
128753,2022,3738.0
148264,2023,3944.0


In [39]:
rows_removed += all_data[all_data['grouped_counterpoint_id'].isna()].shape[0]

In [40]:
all_data = all_data[~all_data['grouped_counterpoint_id'].isna()]

In [41]:
group_columns = ['year', 'grouped_counterpoint_id', 'geocoded_city', 'geocoded_province', 'start_datetime_dayofmonth', 
                                             'count_session_label']


In [42]:
# confirm that there is no NA values in the grouped data - want this to be False
np.any(all_data[group_columns].isna())

np.True_

In [43]:
# where is the NA
all_data[np.any(all_data[group_columns].isna(), axis = 1)][group_columns]

,year,grouped_counterpoint_id,geocoded_city,geocoded_province,start_datetime_dayofmonth,count_session_label
148287,2023,4448.0,Winnipeg,NaN,6,weekday_morning


In [44]:
# manually assign province in this case
all_data.loc[(all_data['year'] == 2023) 
             & (all_data['grouped_counterpoint_id'] == 4448)
             & (all_data['count_session_label'] == 'weekday_morning'), 'geocoded_province'] = 'Manitoba'

In [45]:
# check again
assert not np.any(all_data[group_columns].isna())

### Summary cleaned stats

In [46]:
for year in [2021, 2022, 2023, 2024]:
    if year in [2021, 2022]:
            
        tot_bikes = all_data[all_data['year'] == year].shape[0]
    else:
        tot_bikes = all_data[all_data['year'] == year]['easy_bike_total'].sum()
    tot_locations = all_data[all_data['year'] == year]['counterpoint_id'].value_counts().shape[0]
    tot_communities = all_data[all_data['year'] == year]['geocoded_city'].unique().shape[0]
    tot_minutes = round(all_data[all_data['year'] == year][['survey_session_id',
                                                          'elapsed_time_seconds']].drop_duplicates()['elapsed_time_seconds'].sum() / 60, 1)
    tot_hours = round(tot_minutes / 60, 1)
    print(year, tot_bikes, " bikes counted in ", tot_locations, " locations in ", tot_communities, " communities and ", tot_minutes, " minutes")

2021 79735  bikes counted in  326  locations in  54  communities and  70084.0  minutes
2022 49131  bikes counted in  258  locations in  39  communities and  50533.0  minutes
2023 28503  bikes counted in  187  locations in  34  communities and  31426.8  minutes
2024 47215  bikes counted in  211  locations in  31  communities and  41423.0  minutes


In [47]:
# do aggregation iteratively to have more control

gender_vals = all_data['gender'].dropna().unique()
race_vals = all_data['race'].dropna().unique()
age_vals = all_data['age'].dropna().unique()
names_list = []
bike_total = []
personal_vehicle_total = []
pedestrian_total = []
oversized_total = []
num_sessions_grouped = []
total_time_seconds = []
unique_counterpoint_ids = []
# create empty lists to store results
# note that doing [[]]*N creates N copies of the *same* list
gender_lists = [ [] for _ in range(len(gender_vals))]
age_lists = [ [] for _ in range(len(age_vals))]
race_lists = [ [] for _ in range(len(race_vals))]


for names, group in tqdm(all_data.groupby(group_columns)):
    #if names[2] == 'Mississauga' and names[0] == 2023:
    #    break
    try:
        group['easy_personal_vehicle_total'] = pd.to_numeric(group['easy_personal_vehicle_total']).astype('int')
        personal_vehicle_total.append(np.sum(group['easy_personal_vehicle_total']))
    except:
        personal_vehicle_total.append(np.nan)
        
    try:
        group['easy_pedestrian_total'] = pd.to_numeric(group['easy_pedestrian_total']).astype('int')
        pedestrian_total.append(np.sum(group['easy_pedestrian_total']))
    except:
        pedestrian_total.append(np.nan)

    try:
        group['easy_oversized_total'] = pd.to_numeric(group['easy_oversized_total']).astype('int')
        oversized_total.append(np.sum(group['easy_oversized_total']))
    except:
        oversized_total.append(np.nan)
    
    #group['easy_pedestrian_total'] = group['easy_pedestrian_total'].astype('int', errors = 'ignore')
    #group['easy_oversized_total'] = group['easy_oversized_total'].astype('int', errors = 'ignore')
    
    # demographics
    # still appending to the same list for some reason...
    for i, gender in enumerate(gender_vals):
        gender_lists[i].append(np.sum(group['gender'] == gender))
    
    for i, race in enumerate(race_vals):
        race_lists[i].append(np.sum(group['race'] == race))
    
    for i, age in enumerate(age_vals):
        age_lists[i].append(np.sum(group['age'] == age))
    
    names_list.append(names)
    bike_total.append(group['easy_bike_total'].sum())
    
    unique_counterpoint_ids.append(group['counterpoint_id'].nunique())
    num_sessions_grouped.append(group['survey_session_id'].nunique())
    # note that this takes the bounding times of all the grouped counts INSTEAD of adding up the times for the grouped survey sessions
    # the latter would break if multiple people were counting at a site
    total_time_seconds.append((pd.to_datetime(group['end_datetime']).max() - pd.to_datetime(group['start_datetime']).min()).seconds)
    # FIXME add other vehicle modes, currently there's an entry that's breaking it
    

100%|██████████████████████████████████████| 1859/1859 [00:09<00:00, 196.66it/s]


In [48]:
agg_counts = pd.DataFrame()
agg_counts[group_columns] = names_list
agg_counts['bike_total'] = bike_total
agg_counts['pedestrian_total'] = pedestrian_total
agg_counts['personal_vehicle_total'] = personal_vehicle_total
agg_counts['oversized_vehicle_total'] = oversized_total
agg_counts['num_sessions_grouped'] = num_sessions_grouped
agg_counts['unique_counterpoint_ids'] = unique_counterpoint_ids
agg_counts['total_time_seconds'] = total_time_seconds

for i, gender in enumerate(gender_vals):
    agg_counts['{}_gender_total'.format(gender)] = gender_lists[i]
    
for i, age in enumerate(age_vals):
    agg_counts['{}_age_total'.format(age)] = age_lists[i]
    
for i, race in enumerate(race_vals):
    agg_counts['{}_race_total'.format(race)] = race_lists[i]

In [49]:
# check that all modes got converted to numbers
assert agg_counts[(agg_counts['pedestrian_total'].isna())
           & (agg_counts['year'] > 2022)].shape[0] == 0
assert agg_counts[(agg_counts['personal_vehicle_total'].isna())
           & (agg_counts['year'] > 2022)].shape[0] == 0
assert agg_counts[(agg_counts['oversized_vehicle_total'].isna())
           & (agg_counts['year'] > 2022)].shape[0] == 0

In [50]:
agg_counts.shape

(1859, 29)

In [51]:
# number of resulting count sessions that involved grouping two or more sessions
agg_counts['num_sessions_grouped'].value_counts().iloc[1:].sum()

np.int64(415)

In [52]:
# do some tests

gender_cols = []
for i, gender in enumerate(gender_vals):
    gender_cols.append('{}_gender_total'.format(gender))
    
age_cols = []
for i, age in enumerate(age_vals):
    age_cols.append('{}_age_total'.format(age))
    
race_cols = []
for i, race in enumerate(race_vals):
    race_cols.append('{}_race_total'.format(race))

In [53]:
# each demographic column should add up to the bike total
assert np.all(np.sum(agg_counts[~agg_counts['year'].isin([2023,2024])][gender_cols], axis =1) 
              == agg_counts[~agg_counts['year'].isin([2023,2024])]['bike_total'])
assert np.all(np.sum(agg_counts[~agg_counts['year'].isin([2023,2024])][age_cols], axis =1) 
              == agg_counts[~agg_counts['year'].isin([2023,2024])]['bike_total'])
assert np.all(np.sum(agg_counts[~agg_counts['year'].isin([2023,2024])][race_cols], axis =1)
              == agg_counts[~agg_counts['year'].isin([2023,2024])]['bike_total'])

In [54]:
# check that total bike count adds up
assert all_data['easy_bike_total'].sum() == agg_counts['bike_total'].sum()

In [55]:
# combine columns doubly defined
agg_counts['Unsure_age_total'] = agg_counts['Unsure_age_total'] + agg_counts['Other or unsure_age_total']
agg_counts['Unsure_race_total'] = agg_counts['Other or unsure_race_total']
agg_counts['Unsure_gender_total'] = agg_counts['Other or unsure_gender_total']

agg_counts = agg_counts.drop(columns = ['Other or unsure_race_total', 'Other or unsure_age_total', 'Other or unsure_gender_total'])

In [56]:
# check that total bike count adds up
assert all_data['easy_bike_total'].sum() == agg_counts['bike_total'].sum()

In [57]:
# save aggregated data for analysis
agg_counts.to_csv("data/aggregated_data.csv", index = None)